# Useful Links

Documentation: https://dev.meteostat.net/api

Using API: https://rapidapi.com/meteostat/api/meteostat/playground/apiendpoint_a5f43bba-981c-437f-9dd0-d0c7923ecf4c

List of all weather stations: https://github.com/meteostat/weather-stations?tab=readme-ov-file
- Gives the station ID (function created later to find the closest station)

# Endpoint
- Daily data, used in this notebook
- Provides historical weather statistics for a specific weather station
- Can be queried for a max of 10 years
- Limited to 500 requests per month (on free plan)
- Returns a dictionary of weather data, where each entry is one day at a specific station

In [3]:
import pandas as pd
import requests
import requests_cache
import time

# 1. Defining API Key

In [4]:
def read_key(keyfile):
    with open(keyfile) as f:
        return f.readline().strip("\n")

key = read_key("Meteostat API Key.txt") # this file (in gitignore) holds my personal API key

# 2. Initial Request - Understand Structure of Call

In [5]:
# getting daily weather data from station 10637 from 12/1/2024 - 3/1/2025

session = requests_cache.CachedSession('weather_cache') # caching to prevent duplicate requests to server (only get 500 requests per month with free account)

url = "https://meteostat.p.rapidapi.com/stations/daily" # endpoint

querystring = {"station":"10637","start":"2024-12-01","end":"2025-03-01"} # defining parameters, station = station ID found in the file provided on GitHub, quering over specific time frame

headers = {
	"x-rapidapi-key": key,
	"x-rapidapi-host": "meteostat.p.rapidapi.com"
}

response = session.get(url, headers=headers, params=querystring) # making API call

response.json() # converting response to json, showing the structure of the result

{'meta': {'generated': '2026-03-01 21:26:08'},
 'data': [{'date': '2024-12-01 00:00:00',
   'tavg': -1.4,
   'tmin': -4.1,
   'tmax': 0.0,
   'prcp': 0.3,
   'snow': 0.0,
   'wdir': None,
   'wspd': 6.8,
   'wpgt': 20.9,
   'pres': 1027.9,
   'tsun': 0},
  {'date': '2024-12-02 00:00:00',
   'tavg': 2.5,
   'tmin': 0.0,
   'tmax': 4.3,
   'prcp': 0.4,
   'snow': 0.0,
   'wdir': None,
   'wspd': 4.3,
   'wpgt': 15.8,
   'pres': 1020.0,
   'tsun': 0},
  {'date': '2024-12-03 00:00:00',
   'tavg': 3.8,
   'tmin': 0.0,
   'tmax': 8.0,
   'prcp': 0.0,
   'snow': 0.0,
   'wdir': None,
   'wspd': 10.4,
   'wpgt': 38.2,
   'pres': 1020.4,
   'tsun': 0},
  {'date': '2024-12-04 00:00:00',
   'tavg': 2.2,
   'tmin': -2.7,
   'tmax': 6.2,
   'prcp': 0.0,
   'snow': 0.0,
   'wdir': None,
   'wspd': 7.9,
   'wpgt': 22.3,
   'pres': 1027.8,
   'tsun': 108},
  {'date': '2024-12-05 00:00:00',
   'tavg': 1.1,
   'tmin': -2.1,
   'tmax': 4.7,
   'prcp': 11.1,
   'snow': 0.0,
   'wdir': None,
   'wspd': 14.

# 3. Loading json File with Weather Station Data
- Need this file to get the station id, which is the identifier for specific stations

In [6]:
df_stations = pd.read_json("List of All Weather Stations.json")

In [7]:
# example of one station, used in example call above

df_stations[df_stations["id"] == "10637"]

,id,name,country,region,identifiers,location,timezone,inventory
2565,10637,"{'de': 'Frankfurt Flughafen', 'es': 'Aeropuert...",DE,HE,"{'national': '01420', 'wmo': '10637', 'icao': ...","{'latitude': 50.05, 'longitude': 8.6, 'elevati...",Europe/Berlin,"{'model': {'start': '2018-01-28', 'end': '2026..."


# 4. Defining Distance Function
- Need a function to calculate the distance between an inputted airport and a given station
- Going to match station and airport based on the minimum distance

In [8]:
import math

# defining function that calculates the distance between the latitude/longitude of a two stations

def haversine(air_lat, air_long, search_lat, search_long):
    R = 6371  # Earth radius in kilometers

    # convert degrees to radians
    air_lat = math.radians(air_lat)
    air_long = math.radians(air_long)
    search_lat = math.radians(search_lat)
    search_long = math.radians(search_long)

    dlat = search_lat - air_lat
    dlon = search_long - air_long

    # distance equation between two geographic coordinates
    a = math.sin(dlat / 2)**2 + math.cos(air_lat) * math.cos(search_lat) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c  # returns distance in kilometers

# 5. Defining Function to Calculate Minimum Distance
- Input the latitude and longitude of a location to find the closest weather station
- Returns a tuple with the station id of the closest station, and the distance in km

In [9]:
# defining a function that finds the closest weather station to the inputted latitude and longitude

def find_closest_station(air_lat, air_long):

    # initializing minimum distance variable to compare in the loop
    min_distance = float("inf")
    air_match = None

    # looping through each row in list of weather stations
    for index, station in df_stations.iterrows(): # one station
        location = station["location"] # grabbing the location column which holds latitude and longitude
        search_lat = location.get('latitude') # grabbing the latitude of given weather station
        search_long = location.get('longitude') # grabbing longitude of given weather station

        if search_lat is not None and search_long is not None:
            curr_distance = haversine(air_lat, air_long, search_lat, search_long)

            if curr_distance < min_distance: # if the distance of the current airport is the smallest so far
                min_distance = curr_distance # reassigning the current airport distance to min distance
                air_match = station["id"] # grabbing the station id of the airport
    
    return air_match, min_distance

In [10]:
# testing function on ORD coordinates

air_lat = 41.978611
air_long = -87.904724

station_match, min_distance = find_closest_station(air_lat, air_long)

In [11]:
station_match

'72530'

In [12]:
min_distance

1.1188353396301887

# 6. Finding Station IDs for All Airports

Making comparisons across 6 airports:
- EWR: Newark Liberty International Airport
- BOS: Boston Logan International Airport
- LGA: LaGuardia Airport
- SFO: San Francisco International Airport
- DFW: Dallas/Fort Worth International Airport
- ORD: O’Hare International Airport

These airports were selected based on a list of airports with the most cancellations.

In [13]:
# creating a dictionary of each airport we want weather data on and location info as a tuple (latitude, longitude)

airport_dict = {
    "EWR": (40.689491, -74.174538), 
    "BOS": (42.365589, -71.010025), 
    "LGA": (40.776863, -73.874069),
    "SFO": (37.615223, -122.389977),
    "DFW": (32.897480, -97.040443),
    "ORD": (41.978611, -87.904724)
}

In [14]:
station_dict = {} # initializing a dictionary to hold the airport code and the station id that is matched (based on the minimum distance)
min_distance_dict = {} # initializing a dictionary to hold the airport code and the minimum distance calculated (between the airport and the matched station)

for key, value in airport_dict.items(): # looping through the airports we want to collect weather data on
    air_lat, air_long = value # unpacking the tuple of latitude and longitude
    station_match, min_distance = find_closest_station(air_lat, air_long) # calling the find_closest_station function to get the station id match
    station_dict[key] = station_match # populating the dictionary with airport code and matched station
    min_distance_dict[key] = min_distance # populating the dictionary with airport code and calculated minimum distance

In [15]:
print(station_dict)

{'EWR': '72502', 'BOS': '72509', 'LGA': '72503', 'SFO': '72494', 'DFW': '72259', 'ORD': '72530'}


In [16]:
print(min_distance_dict) # keeping these values to ensure matches make sense

{'EWR': 0.889933073203419, 'BOS': 0.5621389854494424, 'LGA': 0.5629654896408587, 'SFO': 2.0567983694546945, 'DFW': 0.7233679653665868, 'ORD': 1.1188353396301887}


# 7. Validation - Ensuring Accurate Match
- Indexing the weather stations df to ensure the station id found is accurate for each airport

In [17]:
for key, value in station_dict.items():
    
    row = df_stations[df_stations["id"] == value]
    station_name = row.iloc[0]["name"]["en"]
    print(f"Airport: {key}, Station name: {station_name}")

Airport: EWR, Station name: Newark Airport
Airport: BOS, Station name: Boston Logan International
Airport: LGA, Station name: LaGuardia Airport
Airport: SFO, Station name: San Francisco Airport
Airport: DFW, Station name: Dallas/Ft. Worth International  Airport
Airport: ORD, Station name: Chicago O’hare Airport


All of the matches look accurate, ready to call the API to grab weather data for each location

# 8. Creating Dictionary to Hold All Weather Data for All Locations

In [18]:
session = requests_cache.CachedSession('weather_cache')

url = "https://meteostat.p.rapidapi.com/stations/daily"

weather_dict = {}

for airport, station_id in station_dict.items():

    querystring = {"station":station_id,"start":"2024-12-01","end":"2025-02-28"}

    # need to hid my api key
    headers = {
        "x-rapidapi-key": key,
        "x-rapidapi-host": "meteostat.p.rapidapi.com"
    }

    response = session.get(url, headers=headers, params=querystring)

    if response.status_code == 200:
    
        weather_result = response.json()["data"] # weather data within an outside dictionary, only want to store this inner dictionary
        weather_dict[airport] = weather_result

    time.sleep(10)

In [19]:
weather_dict # showing the structure of the dictionary, airport is a key in the outer value with an inner dictionary to hold daily weather data

{'EWR': [{'date': '2024-12-01 00:00:00',
   'tavg': -0.2,
   'tmin': -4.3,
   'tmax': 3.9,
   'prcp': 0.0,
   'snow': 0.0,
   'wdir': None,
   'wspd': 16.2,
   'wpgt': None,
   'pres': 1018.9,
   'tsun': None},
  {'date': '2024-12-02 00:00:00',
   'tavg': 0.7,
   'tmin': -3.8,
   'tmax': 5.6,
   'prcp': 0.0,
   'snow': 0.0,
   'wdir': None,
   'wspd': 13.3,
   'wpgt': None,
   'pres': 1020.8,
   'tsun': None},
  {'date': '2024-12-03 00:00:00',
   'tavg': 2.5,
   'tmin': 0.0,
   'tmax': 6.7,
   'prcp': 0.0,
   'snow': 0.0,
   'wdir': None,
   'wspd': 15.8,
   'wpgt': None,
   'pres': 1023.9,
   'tsun': None},
  {'date': '2024-12-04 00:00:00',
   'tavg': 1.1,
   'tmin': -4.3,
   'tmax': 5.6,
   'prcp': 0.3,
   'snow': 0.0,
   'wdir': None,
   'wspd': 16.9,
   'wpgt': None,
   'pres': 1019.9,
   'tsun': None},
  {'date': '2024-12-05 00:00:00',
   'tavg': 3.4,
   'tmin': -0.5,
   'tmax': 5.0,
   'prcp': 1.5,
   'snow': 0.0,
   'wdir': None,
   'wspd': 28.8,
   'wpgt': None,
   'pres': 1004

# 9. Data Wrangling of weather_dict
- Want a dataframe for each location, where each of the days weather data is collected is a row
- Columns with date, and all weather-related information

In [20]:
list_df = [] # initializing a list to hold each individual df

for airport, info in weather_dict.items(): # looping through the weather dictionary to convert each airport's dictionary to a dataframe
    df = pd.DataFrame(info) # converting dictinary[info] to dataframe - info holds all the weather info, don't need the other stuff in the dictionary
    df["airport"] = airport # adding a column for the airport name
    list_df.append(df) # adding dictionary to list

In [21]:
weather_df = pd.concat(list_df, ignore_index=True, sort=False) # concatenating all of the dataframes created by the loop into one big dataframe
weather_df = weather_df.convert_dtypes() # using this to ensure any missing values are coded properly as NA
weather_df.to_csv("Airport Weather Data.csv", index=False) # saving df as csv

/var/folders/7_/m2xp0d794mx60kj_cp1dk6180000gn/T/ipykernel_8217/3650422393.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  weather_df = pd.concat(list_df, ignore_index=True, sort=False) # concatenating all of the dataframes created by the loop into one big dataframe
